# H SC 790 · Lab 9 · Build a Cohort Someone Else Could Rebuild

**Week 10.** Due Thursday, April 8, 11:59 PM · 30 points.

**What earns full credit**
- A fresh run reproduces count and estimate exactly
- Naive and aligned estimates both shown
- Selection and misclassification quantified
- AI-use note included

**Rules.** Everything here is synthetic. Never load protected health information or confidential employer data into Colab or GitHub. The lab is finished when it runs top to bottom from a fresh runtime (Runtime → Restart session and run all).

The data dictionary for this lab is the README in the lab's folder of the course repository.

Cells marked `# YOUR CODE HERE` and *Your answer here.* are yours to complete. Add cells freely.

In [ ]:
# Setup: run this cell first, every time.
import os, json, urllib.request
import numpy as np
import pandas as pd

# The course data, read straight from the course repository. Never from your own machine.
COURSE_URL = os.environ.get("HSC790_DATA", "https://raw.githubusercontent.com/OWNER/hsc790-spr27/main/data/")

def read_text(rel):
    """Text of one file in the course data folder, for example read_text('SNAPSHOT.txt')."""
    return urllib.request.urlopen(COURSE_URL + rel).read().decode()

print("Data snapshot:", read_text("SNAPSHOT.txt").strip())

**The question.** Among adults with type 2 diabetes at Puente County Health Services, does enrolling in the Diabetes Care Management Program reduce emergency department visits within 12 months?

You have the Lab 3 EHR extract (`lab03/`) and the program's enrollment file (`lab09/program_enrollment.csv`). The program began as a pilot taking hospital-discharge referrals in September 2024 and opened to primary care referrals on January 1, 2025. Emulate the trial you would have liked to run, and make the cohort something another analyst could rebuild exactly.

In [ ]:
import hashlib
SNAPSHOT = read_text("SNAPSHOT.txt").strip()
pt = pd.read_csv(COURSE_URL + "lab03/patients.csv", parse_dates=["birth_date", "first_seen", "last_seen"], dtype={"zip": str})
enc = pd.read_csv(COURSE_URL + "lab03/encounters.csv.gz", parse_dates=["encounter_date", "end_date"])
cond = pd.read_csv(COURSE_URL + "lab03/conditions.csv.gz", parse_dates=["onset_date"])
obs = pd.read_csv(COURSE_URL + "lab03/observations.csv.gz", parse_dates=["obs_date"])
prog = pd.read_csv(COURSE_URL + "lab09/program_enrollment.csv", parse_dates=["enroll_date"])
print(SNAPSHOT, "|", len(pt), "patients |", len(prog), "enrollments")
prog.groupby([prog.enroll_date.dt.to_period("Q"), "referral_source"]).size().unstack(fill_value=0)

## 1. Write the target trial

Fill in the table before you query anything.

| Component | Target trial | Emulation with these data |
|---|---|---|
| Eligibility | *Your answer here.* | *Your answer here.* |
| Treatment strategies | *Your answer here.* | *Your answer here.* |
| Assignment | *Your answer here.* | *Your answer here.* |
| Time zero | *Your answer here.* | *Your answer here.* |
| Outcome | *Your answer here.* | *Your answer here.* |
| Follow-up | *Your answer here.* | *Your answer here.* |
| Causal contrast | *Your answer here.* | *Your answer here.* |

## 2. Build the cohort from a criteria list

Build one row per candidate patient with time zero and every variable the criteria need. Then let the slide's loop apply the criteria and produce the attrition table. Record the data snapshot identifier with it.

In [ ]:
# YOUR CODE HERE

# --- cohort as a criteria list; attrition for free (slide) ---
criteria = [('Age 18+',            'age >= 18'),
            ('Type 2 diabetes',    't2dm'),
            ('Visit in prior year','prior_visits >= 1'),
            ('Not yet enrolled',   'not prior_enrolled'),
            ('Baseline A1c',       'a1c == a1c')]  # not NaN
cohort, rows = base, [('All patients', len(base))]
for name, rule in criteria:
    cohort = cohort.query(rule)
    rows.append((name, len(cohort)))
attrition = pd.DataFrame(rows, columns=['step', 'n'])
attrition["lost"] = -attrition.n.diff().fillna(0).astype(int)
print("data snapshot:", SNAPSHOT)
attrition

## 3. Run it wrong, then right

First classify people by whether they *ever* enrolled during follow-up and compare 12-month risk. Then align time zero: give everyone a 30-day grace period, start follow-up at day 30 for people with no emergency visit before then, and classify by enrollment status at day 30. Adjust the aligned comparison for confounders. Explain the difference.

In [ ]:
def risk_ratio(df, exposed, outcome):
    a, b = df[df[exposed]][outcome].mean(), df[~df[exposed]][outcome].mean()
    return pd.Series({"n_exposed": int(df[exposed].sum()), "risk_exposed": round(a, 3), "risk_unexposed": round(b, 3), "risk_ratio": round(a / b, 2)})

# YOUR CODE HERE

In [ ]:
# adjust the aligned comparison for measured confounders, by standardization
lm = c[~c.ed_0_30].assign(by30=lambda d: d.enroll_day.between(0, 30))
crude = risk_ratio(lm, "by30", "ed_30_365")
X = pd.DataFrame({"age": lm.age, "prior_visits": lm.prior_visits.clip(upper=12), "prior_ed": lm.prior_ed.clip(upper=5), "a1c": lm.a1c, "lep": lm.lep})
m = LogisticRegression(C=1e6, max_iter=5000).fit(X.assign(t=lm.by30.astype(int)), lm.ed_30_365)
p1, p0 = m.predict_proba(X.assign(t=1))[:, 1].mean(), m.predict_proba(X.assign(t=0))[:, 1].mean()
print("landmark, crude:", crude.to_dict())
print("landmark, adjusted: risk", round(p1, 3), "vs", round(p0, 3), "-> risk ratio", round(p1 / p0, 2))
print("\nmedian days from time zero to enrollment:", c.enroll_day[c.enroll_day.between(0, 365)].median(), "| enrolled after day 30:", int((c.enroll_day > 30).sum()))
w = c[(c.enroll_day > 0) | c.enroll_day.isna()]
print("share who ever enrolled, by whether they had an emergency visit in the first 30 days:")
print(w.groupby("ed_0_30").apply(lambda g: g.enroll_day.between(0, 365).mean(), include_groups=False).round(3).to_string())

## 4. Examine selection

Find the largest loss in your attrition table. Compare those kept with those lost. Then re-estimate without that criterion, or with weights that make the kept look like everyone eligible.

In [ ]:
# YOUR CODE HERE

## 5. Correct for misclassification

Your cohort depends on a phenotype, and your outcome on what the extract can see. Use the slide's correction with your Lab 3 sensitivity and specificity, by clinic, to estimate how many people with diabetes the cohort definition misses. Then vary the sensitivity of the outcome (the extract sees only PCHS's own emergency department) and report how the estimate moves.

In [ ]:
# --- correct an observed proportion (slide) ---
def corrected(p_obs, se, sp):
    return (p_obs + sp - 1) / (se + sp - 1)

# YOUR CODE HERE

## 6. Prove it reproduces

Fix seeds, pin versions, and print a checksum of the final cohort. A fresh run must match exactly.

In [ ]:
# --- one string that proves the cohort is identical (slide) ---
ids = ','.join(map(str, sorted(cohort.patient_id)))
print(hashlib.sha256(ids.encode()).hexdigest()[:12])

# YOUR CODE HERE

## Interpretation

Cite your own output in each answer.

**1. How much did time zero matter?** Give both estimates and say which analysis a reader would have believed.

*Your answer here.*

**2. Who did your cohort leave out?** For the largest loss, how did those lost differ, and which way does that bias the result?

*Your answer here.*

**3. How robust is the estimate?** Report it across your range of sensitivity and specificity.

*Your answer here.*

**4. Is this research?** If your organization ran this analysis, who would need to approve it, and why?

*Your answer here.*

## AI-use note (required)

Say which AI tools you used, for what, and one thing you checked or corrected in their output. "None" is an acceptable answer.

*Your note here.*

## Submit

1. Runtime → Restart session and run all. Fix anything that fails.
2. File → Save a copy in GitHub, into `labs/lab09/` of your course repository.